In [1]:
# Import packages
import os
import matplotlib.pyplot as plt
import pandas as pd
import re
import numpy as np
import glob
from pathlib import Path
from scipy import sparse
from copy import deepcopy
import csv
import itertools
import warnings
import sys
import scanpy as sc

In [41]:
# Set directories
out_dir = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/postprocess_adata/'
os.makedirs(out_dir,exist_ok=True)

In [42]:
# This is the sample description text file
sample_df = pd.read_csv('/data/chanjlab/forsythb/organoid_analysis_pipeline_scripts/' + 'organoid_sample_description.txt', sep='\t', index_col = 0)
sample_df.head()

,Patient,Tumor_Site,Culture_Media,ZFP_Expression,Replicate,Batch,Sample
Original_Sample_Name,,,,,,,
146P_BASE_shCTRL,146,Primary,BASE,CTRL,1,5,146_P_BASE_CTRL_1
146P_BASE_shZFP36L2_3,146,Primary,BASE,ZFP_KD,1,5,146_P_BASE_ZFPKD_1
146P_BASE_shZFP36L2_4,146,Primary,BASE,ZFP_KD,2,5,146_P_BASE_ZFPKD_2
146P_HISC_shCTRL,146,Primary,HISC,CTRL,1,6,146_P_HISC_CTRL_1
146P_HISC_shZFP36L2_3,146,Primary,HISC,ZFP_KD,1,6,146_P_HISC_ZFPKD_1


In [43]:
# These are the output files from scran
mtx_fn = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/scran/' +  'scran.matrix.mtx'
g_fn = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/scran/' + 'scran.genes.csv'
bc_fn = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/scran/' + 'scran.barcodes.csv'

In [44]:
# Read the scran output files
fn = g_fn
with open(fn, 'r') as f:
    g = [i.strip() for i in f.readlines()]

fn = bc_fn
with open(fn, 'r') as f:
    bc = [i.strip() for i in f.readlines()]

from scipy.io import mmread
norm_df = mmread(mtx_fn)
norm_df = norm_df.tocsr().toarray()
norm_df = pd.DataFrame(norm_df, index=bc, columns=g)

samples = [re.sub('_[ACGT]+-1$','',i) for i in bc]
samples = pd.Series(samples, index = bc)

In [45]:
# This is the concatenated adata
adata = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/concatenated/adata.combined.h5ad')

bc_intersect = adata.obs.index.intersection(norm_df.index)
g_intersect = adata.var_names.intersection(norm_df.columns)
adata = adata[bc_intersect, g_intersect]
adata.layers["counts"] = adata.X.copy()
norm_df = norm_df.loc[adata.obs.index, adata.var_names]

adata.X = norm_df.values
adata.layers['without_log'] = adata.X

adata.raw = adata  # Save the raw data before log transformation
sc.pp.log1p(adata, base=2)

sample_df2 = sample_df.loc[samples,:]
sample_df2.index = adata.obs.index

adata.obs = pd.concat([adata.obs.drop('Sample', axis=1), sample_df2], axis=1)

adata.obs['Sample'] = adata.obs['Sample'].astype('category')

/scratch/lsftmp/3831838.tmpdir/ipykernel_78954/2525911331.py:7: ImplicitModificationWarning: Setting element `.layers['counts']` of view, initializing view as actual.
  adata.layers["counts"] = adata.X.copy()


In [46]:
'''
HVGs
'''
adata.obs['Batch'] = adata.obs.Batch.astype(str).astype('category')

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=5000,
    layer="counts",
    flavor="seurat_v3",  # Change to "seurat"
    batch_key="Batch"
)

In [47]:
# Remove bad genes
norm_df = pd.DataFrame(adata.X, index=adata.obs_names, columns = adata.var_names)
norm_df = norm_df.loc[:,adata.var.highly_variable]

bad_genes = norm_df.columns.str.contains(
    "^MT-|^MTMR|^MTND|NEAT1|TMSB4X|TMSB10|^RPS|^RPL|^MRP|^FAU$|UBA52|MALAT")
norm_df = norm_df.loc[:,~bad_genes]

In [48]:
# Functions for PCA
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA

import numpy.matlib
def kneepoint(vec):
    curve =  [1-x for x in vec]
    nPoints = len(curve)
    allCoord = np.vstack((range(nPoints), curve)).T
    np.array([range(nPoints), curve])
    firstPoint = allCoord[0]
    lineVec = allCoord[-1] - allCoord[0]
    lineVecNorm = lineVec / np.sqrt(np.sum(lineVec**2))
    vecFromFirst = allCoord - firstPoint
    scalarProduct = np.sum(vecFromFirst * numpy.matlib.repmat(lineVecNorm, nPoints, 1), axis=1)
    vecFromFirstParallel = np.outer(scalarProduct, lineVecNorm)
    vecToLine = vecFromFirst - vecFromFirstParallel
    distToLine = np.sqrt(np.sum(vecToLine ** 2, axis=1))
    idxOfBestPoint = np.argmax(distToLine)
    return idxOfBestPoint

def RunPCA(cts, var_threshold, n_components=300):
    pca = PCA(n_components=n_components, svd_solver='randomized')
    pca.fit(cts)
    num_components = 0
    num_components = max(num_components,kneepoint(np.cumsum(pca.explained_variance_ratio_)))
    num_components = max(num_components,np.where(np.cumsum(pca.explained_variance_ratio_) > var_threshold)[0][0])
    var_explained = np.cumsum(pca.explained_variance_ratio_)[num_components]
    print('# Components = %d' % (num_components+1))
    print('Variance explained = %f' % var_explained)
    return pca, num_components, var_explained

In [49]:
'''
PCA
'''
print('Performing PCA')
n_components=500
pca = PCA(n_components=n_components, svd_solver='randomized')
pca.fit(norm_df)

#By Kneepoint
num_components = 0
num_components = max(num_components,kneepoint(np.cumsum(pca.explained_variance_ratio_)))
print('# Components = %d' % (num_components+1))

var_explained = np.cumsum(pca.explained_variance_ratio_)[num_components]
print('Variance explained = %f' % var_explained)

pca = PCA(n_components=num_components, svd_solver='randomized')
pca_merge = pd.DataFrame(pca.fit_transform(norm_df.values),
                index=norm_df.index)
adata.obsm['X_pca'] = pca_merge.loc[adata.obs_names,:].values
adata.uns['num_components'] = num_components
adata.uns['var_explained'] = var_explained

Performing PCA
# Components = 54
Variance explained = 0.369261


In [50]:
'''
NEAREST NEIGHBORS
'''
print('Performing nearest neighbors')
n_neighbors=30
min_dist = 0.3
sc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=pca_merge.shape[1])

Performing nearest neighbors


In [51]:
'''
CLUSTERING
'''
print('Phenograph Clustering')
import phenograph
clusters_merge, _, _ = phenograph.cluster(pca_merge, k=30)
clusters_merge = pd.Series(clusters_merge, pca_merge.index)

adata.obs['phenograph'] = clusters_merge.loc[adata.obs_names].astype('str').astype('category')

Phenograph Clustering
Finding 30 nearest neighbors using minkowski metric and 'auto' algorithm
Neighbors computed in 3.0972321033477783 seconds
Jaccard graph constructed in 12.164788007736206 seconds
Wrote graph to binary file in 0.609992265701294 seconds
Running Louvain modularity optimization
After 1 runs, maximum modularity is Q = 0.912614
Louvain completed 21 runs in 30.60321307182312 seconds
Sorting communities by size, please wait ...
PhenoGraph completed in 51.33668303489685 seconds


In [52]:
'''
LEIDEN CLUSTERING
'''
print('Leiden Clustering')
sc.tl.leiden(adata, resolution = 1.8)

Leiden Clustering


In [53]:
'''
UMAP
'''
print('Performing UMAP')
sc.tl.paga(adata, groups = 'phenograph')
sc.pl.paga(adata, plot=False)

sc.tl.umap(adata, init_pos='paga', min_dist=min_dist)

Performing UMAP


In [54]:
'''
DIFFUSION MAP
'''
print('Performing Diffusion Map')
sc.tl.diffmap(adata, n_comps = 20)

Performing Diffusion Map


In [55]:
'''
DEG
'''
stats = {}
# Log-transform the count data
#sc.pp.log1p(adata, base=2)

# Perform DEG analysis
print('Performing DEG')
sc.tl.rank_genes_groups(adata, groupby='phenograph', method='wilcoxon')
# stats[group_name, 'logfoldchanges'] = np.log2(
#     pd.concat(stats[group_name, 'scale1'], axis=1)
# )

Performing DEG


/home/forsythb/.local/lib/python3.9/site-packages/numpy/core/fromnumeric.py:86: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)
/home/forsythb/.local/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:396: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, 'names'] = self.var_names[global_indices]
/home/forsythb/.local/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:398: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor 

In [56]:
# Save the processed data
ofile = out_dir + 'adata.combined.postprocess.h5ad'
adata.write_h5ad(ofile)

adata.obs.to_csv(out_dir + 'obs.combined.postprocess.csv', sep ='\t')

In [57]:
adata = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/postprocess_adata/adata.combined.postprocess.h5ad')

In [58]:
adata

AnnData object with n_obs × n_vars = 52800 × 31806
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'log10GenesPerUMI', 'original_total_counts', 'log10_original_total_counts', 'mito_frac', 'Patient', 'Tumor_Site', 'Culture_Media', 'ZFP_Expression', 'Replicate', 'Batch', 'Sample', 'phenograph', 'leiden'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ribo', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'diffmap_evals', 'hvg', 'leiden', 'log1p', 'neighbors', 'num_components', 'paga', 'phenograph_sizes', 'rank_genes_groups', 'umap', 'var_explained'
    obsm: 'X_diffmap', 'X_pca', 'X_

In [59]:
# Identify genes containing 'HTO' or 'TSB'
hto_genes = [gene for gene in adata.var_names if "HTO" in gene]
tsb_genes = [gene for gene in adata.var_names if "TSB0" in gene]

# Remove identified genes
if hto_genes:
    print("Removed genes containing 'HTO':", hto_genes)

if tsb_genes:
    print("Removed genes containing 'TSB':", tsb_genes)

In [61]:
print(hto_genes)
print(tsb_genes)

[]
[]
